<a href="https://colab.research.google.com/github/Precious00001/GenerativeAI-101/blob/main/legal-document-research-assistant/v1.0/Legal_Document_Research_Assistant.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [23]:
!pip install -q langchain langchain-google-genai google-generativeai tiktoken langchain-community
!pip install langchain langchain-google-genai google-generativeai tiktoken faiss-cpu huggingface_hub langchain-community langgraph
!pip install -U langchain-groq
!pip install -U langchain-huggingface sentence-transformers
!pip install -U langchain-community faiss-cpu
!pip install tavily-python
!pip install pypdf
!pip install langchain-text-splitters
!pip install duckduckgo-search
!pip install -U ddgs

In [24]:
from langchain_groq import ChatGroq
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_community.document_loaders import PyPDFLoader
from langchain_community.tools import DuckDuckGoSearchResults
from langchain_core.tools import tool
from langchain_core.documents import Document
from langgraph.prebuilt import create_react_agent
import uuid
import time
from datetime import datetime, timedelta
from google.colab import userdata

In [25]:

#LLM
groq_api_key = userdata.get("your_api_key")
llm = ChatGroq(
    model="llama-3.3-70b-versatile",
    api_key=groq_api_key,
    temperature=0,
    max_tokens=500,
    max_retries=3,  # retry automatically
    request_timeout=60
)

In [26]:
#Searching tool
search_tool = DuckDuckGoSearchResults()

#Vectorstore (will set in main)
vectorstore = None

JURISDICTION_RULES = {
    "federal court": 21,
    "state court": 30,
    "high court": 30,
    "supreme court": 90,
    "magistrate court": 14,
    "appeals court": 30,
    "court of appeal": 28
}

In [27]:
def load_pdfs(pdf_paths):
  all_text = ""
  for path in pdf_paths:
    loader = PyPDFLoader(path)
    documents = loader.load()
    for doc in documents:
        all_text += doc.page_content + "\n"
    print(f"Loaded: {path}")
  return all_text

def chunk_text(text, chunk_size=500):
  chunks = []
  for i in range(0, len(text), chunk_size):
    chunk = text[i:i+chunk_size]
    if len(chunk) > 200:
      chunks.append(chunk)
  return chunks or [text]

# Load embeddings once at the top level, not inside the function
print("Loading embeddings model...")
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
print("Embeddings model loaded!")

def create_vector_store(texts):
# Convert raw text chunks into LangChain Document objects
    documents = [
        Document(
            page_content=text,
            metadata={"id": str(uuid.uuid4())}
        ) for text in texts
    ]

    # Use the already loaded embeddings instead of reloading
    vector_store = FAISS.from_documents(documents, embeddings)

    return vector_store

Loading embeddings model...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings model loaded!


In [28]:
@tool
def calculate_deadline(filing_info: str) -> str:
    """Calculate the response deadline given a filing date and jurisdiction.
    Input format: 'filing_date, jurisdiction'
    Date can be DD/MM/YYYY or YYYY-MM-DD
    Example: '15/01/2026, federal court' or '2026-01-15, federal court'
    """
    try:
        # Split input into filing date and jurisdiction
        parts = filing_info.split(",")
        filing_date_str = parts[0].strip()
        jurisdiction = parts[1].strip().lower()

        # Try multiple date formats
        date_formats = ["%d/%m/%Y", "%Y-%m-%d", "%m/%d/%Y"]
        filing_date = None

        # Loop through formats until one works
        for fmt in date_formats:
            try:
                filing_date = datetime.strptime(filing_date_str, fmt)
                break
            except ValueError:
                continue

        # If no format worked return helpful error
        if filing_date is None:
            return "Invalid date format. Please use DD/MM/YYYY or YYYY-MM-DD"

        # Look up jurisdiction rules
        if jurisdiction not in JURISDICTION_RULES:
            available = ", ".join(JURISDICTION_RULES.keys())
            return f"Jurisdiction '{jurisdiction}' not found. Available: {available}"

        # Calculate deadline
        days_allowed = JURISDICTION_RULES[jurisdiction]
        deadline = filing_date + timedelta(days=days_allowed)

        return (
            f"Source: Deadline Calculator\n\n"
            f"Filing Date: {filing_date.strftime('%B %d, %Y')}\n"
            f"Jurisdiction: {jurisdiction.title()}\n"
            f"Days Allowed: {days_allowed} days\n"
            f"Response Deadline: {deadline.strftime('%B %d, %Y')}"
        )
    except Exception as e:
        return f"Error calculating deadline: {str(e)}"


@tool
def summarize_case(case_name: str) -> str:
    """Search for a structured summary of a legal case.
    Checks loaded PDF documents first, then falls back to internet search.
    Input: case name e.g. 'Roe v Wade'
    IMPORTANT: Call this tool only ONCE per case name.
    """
    try:
        # --- Step 1: Check PDF documents first ---
        if vectorstore is not None:
            docs_with_scores = vectorstore.similarity_search_with_score(case_name, k=3)
            relevant_docs = [doc for doc, score in docs_with_scores if score < 1.5]

            if relevant_docs:
                context = "\n".join([f"- {doc.page_content}" for doc in relevant_docs])
                prompt = f"You are a legal assistant. Summarize the case '{case_name}' with sections: Case Name, Parties Involved, Key Facts, Legal Issues, Outcome. Context: {context}"
                response = llm.invoke(prompt)
                # Return immediately, do not search internet
                return f"Source: Legal PDF Documents\n\n{response.content.strip()}"

        # --- Step 2: Internet search fallback ---
        results = search_tool.invoke({"query": f"{case_name} legal case summary"})
        prompt = f"You are a legal assistant. Summarize the case '{case_name}' with sections: Case Name, Parties Involved, Key Facts, Legal Issues, Outcome. Search Results: {results}"
        response = llm.invoke(prompt)
        # Return immediately after first search, do not retry
        return f"Source: Internet Search\n\n{response.content.strip()}"

    except Exception as e:
        return f"Error summarizing case: {str(e)}"


In [29]:
system_prompt = """You are a legal research assistant with two tools:
1. calculate_deadline - calculates court response deadlines
2. summarize_case - summarizes legal cases

RULES:
- Call each tool ONCE only
- After getting the tool result, immediately give your final answer
- Format: [full answer]\nSource: [exact source label from tool result]
- Never invent source labels, copy them exactly from the tool result"""

def main():
    global vectorstore

    # --- Load and index legal PDF documents ---
    try:
        pdf_paths = ["/content/service-ll-usrep-usrep410-usrep410113-usrep410113.pdf"]  # replace with your PDF path
        pdf_text = load_pdfs(pdf_paths)
        chunks = chunk_text(pdf_text)
        chunks = chunks[:20] # limit during dev to avoid rate limits — remove for production
        vectorstore = create_vector_store(chunks)
        print("Vector store created successfully!")
    except Exception as e:
        print(f"Vector store creation failed: {str(e)}")
        vectorstore = None

    # --- Set up tools and agent ---
    tools = [calculate_deadline, summarize_case]
    agent = create_react_agent(llm, tools, prompt=system_prompt)

    # --- Test questions ---
    questions = [
        "Summarize the case Roe v Wade",
        "If a case was filed on 2026-01-15 in federal court, when is the response due?",
        "What are the key legal issues in Brown v Board of Education?"
    ]

    # --- Run each question through the agent ---
    for question in questions:
        print(f"\nQuestion: {question}")
        response = agent.invoke(
            {"messages": [("human", question)]},
            config={"recursion_limit": 15}
        )
        print(response['messages'][-1].content)
        print("-" * 50)
        time.sleep(2)  # prevent rate limit

main()

Vector store creation failed: File path /content/service-ll-usrep-usrep410-usrep410113-usrep410113.pdf is not a valid file or url

Question: Summarize the case Roe v Wade


/tmp/ipykernel_9688/3407382167.py:28: LangGraphDeprecatedSinceV10: create_react_agent has been moved to `langchain.agents`. Please update your import to `from langchain.agents import create_agent`. Deprecated in LangGraph V1.0 to be removed in V2.0.
  agent = create_react_agent(llm, tools, prompt=system_prompt)


[The Supreme Court ruled in favor of Jane Roe, establishing a woman's right to choose to have an abortion under the Due Process Clause of the Fourteenth Amendment, which was later overturned by the Dobbs v. Jackson Women's Health Organization decision in 2022.]
Source: Internet Search
--------------------------------------------------

Question: If a case was filed on 2026-01-15 in federal court, when is the response due?
[February 05, 2026]
Source: Deadline Calculator
--------------------------------------------------

Question: What are the key legal issues in Brown v Board of Education?
[The key legal issues in Brown v Board of Education were whether segregation in public schools, as allowed by the "separate but equal" doctrine, violated the Equal Protection Clause of the Fourteenth Amendment to the U.S. Constitution, and whether the segregation of public schools based solely on race deprived African American children of equal educational opportunities.]
Source: Internet Search
----